# Phase 6: Systematic System Evaluation Framework

## Music Brain Wellbeing Intelligence System

This notebook demonstrates the Phase 6 Systematic Evaluation Layer. It independently evaluates:
1. **Recommendation Engine (Phase 2 & 3):** Mean Vector Distance, Score Monotonicity, Intra-List Diversity, Cluster Coverage, and Random Baseline Comparison.
2. **Research-Grounded RAG Retriever (Phase 4):** Retrieval Hit Rate @ K, Mean Reciprocal Rank (MRR), and Cosine Distance Distribution.
3. **Grounded LLM Explanation Layer (Phase 5):** JSON Structural Validity, Citation Grounding Accuracy, Track ID Grounding Accuracy, and Non-Clinical Safety Compliance.
4. **End-to-End Pipeline:** Stage-by-Stage Latency Breakdown, Completion Rates, and Validation Pass Rates.

> **Scientific Boundary & Evaluation Discipline:** We explicitly do **not** invent fake ground-truth user ratings or claim metrics like Precision, Recall, or NDCG without user interaction data. Evaluation runs 100% offline in DEMO mode.

In [1]:
import os
import sys
import json
import pandas as pd

# Ensure project root is in python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.music_loader import load_music_catalog
from src.rag.embeddings import EmbeddingModel
from src.rag.vector_store import VectorStore
from src.rag.ingest import IngestionPipeline
from src.rag.retriever import ResearchRetriever
from src.explanation.schemas import ExplanationRequest, SafetyConstraints
from src.explanation.explanation_generator import ExplanationGenerator
from src.evaluation.recommendation_eval import RecommendationEvaluator
from src.evaluation.rag_eval import RAGRetrieverEvaluator
from src.evaluation.llm_eval import LLMExplanationEvaluator
from src.evaluation.end_to_end_eval import EndToEndEvaluator

print("All Phase 6 evaluation module imports successful!")

All Phase 6 evaluation module imports successful!


## 1. Load Controlled Evaluation Benchmark Datasets

We load version-controlled JSON benchmarks from `data/evaluation/`.

In [2]:
profiles_path = os.path.join(project_root, "data", "evaluation", "controlled_profiles.json")
rag_queries_path = os.path.join(project_root, "data", "evaluation", "rag_eval_queries.json")
catalog_path = os.path.join(project_root, "data", "raw", "spotify", "tracks.csv")

with open(profiles_path, "r", encoding="utf-8") as f:
    controlled_profiles = json.load(f)

with open(rag_queries_path, "r", encoding="utf-8") as f:
    rag_queries = json.load(f)

catalog_df = load_music_catalog(catalog_path)

print(f"Loaded {len(controlled_profiles)} controlled user profile personas.")
print(f"Loaded {len(rag_queries)} controlled RAG benchmark queries.")
print(f"Loaded {len(catalog_df)} catalog tracks.")

Loaded 3 controlled user profile personas.
Loaded 4 controlled RAG benchmark queries.
Loaded 500 catalog tracks.


## 2. Recommendation Engine Evaluation

We evaluate recommendation performance across controlled personas and compare vector distance against random baseline recommendations.

In [3]:
rec_evaluator = RecommendationEvaluator()
rec_eval_results = []

for prof in controlled_profiles:
    res = rec_evaluator.evaluate_recommendations(prof, catalog_df, top_n=5)
    res["persona"] = prof["persona"]
    rec_eval_results.append(res)

rec_summary_df = pd.DataFrame(rec_eval_results)[
    ["persona", "mean_feature_distance", "random_baseline_distance", "distance_improvement_over_random", "ranking_monotonicity_score", "intra_list_diversity", "cluster_coverage", "is_outperforming_random"]
]

print("Recommendation Engine Evaluation Summary:")
display(rec_summary_df)

Recommendation Engine Evaluation Summary:


,persona,mean_feature_distance,random_baseline_distance,distance_improvement_over_random,ranking_monotonicity_score,intra_list_diversity,cluster_coverage,is_outperforming_random
0,acoustic_relaxing,43.4987,63.8156,20.3169,1.0,12.8757,0.0,True
1,upbeat_energetic,11.8367,13.0263,1.1896,1.0,12.8757,0.0,True
2,ambient_focus,28.5039,48.8202,20.3163,1.0,12.8757,0.0,True


## 3. RAG Research Retrieval Evaluation

We evaluate top-K semantic retrieval Hit Rate @ K and Mean Reciprocal Rank (MRR) over controlled benchmark queries.

In [4]:
chroma_dir = os.path.join(project_root, "data", "vector_store", "chroma")
jsonl_path = os.path.join(project_root, "data", "raw", "research", "music_wellbeing_research.jsonl")

vector_store = VectorStore(persist_directory=chroma_dir, collection_name="music_wellbeing_research")
embedder = EmbeddingModel()
pipeline = IngestionPipeline(embedder=embedder, vector_store=vector_store)
pipeline.run(jsonl_path)

retriever = ResearchRetriever(embedder=embedder, vector_store=vector_store)
rag_evaluator = RAGRetrieverEvaluator()

rag_res = rag_evaluator.evaluate_retriever(retriever, rag_queries, top_k=2)

print(f"RAG Retrieval Evaluation Summary:")
print(f" - Mean Hit Rate @ K=2: {rag_res['mean_hit_rate_at_k'] * 100:.1f}%")
print(f" - Mean Reciprocal Rank (MRR): {rag_res['mean_mrr']:.4f}")
print(f" - Average Cosine Distance: {rag_res['average_cosine_distance']:.4f}")
print(f" - Empty Query Resilience Pass: {rag_res['empty_query_resilience_pass']}")

print("\nQuery Breakdown Details:")
display(pd.DataFrame(rag_res["query_breakdowns"]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG Retrieval Evaluation Summary:
 - Mean Hit Rate @ K=2: 50.0%
 - Mean Reciprocal Rank (MRR): 0.3750
 - Average Cosine Distance: 0.3012
 - Empty Query Resilience Pass: True

Query Breakdown Details:


,query_id,query_text,retrieved_count,hit_rate_at_k,mrr,top_retrieved_pmid
0,Q_EVAL_001,music therapy stress reduction RCT meta-analysis,2,1.0,1.0,33176590
1,Q_EVAL_002,passive music listening anxiety systematic review,2,1.0,0.5,40547443
2,Q_EVAL_003,sad music listening emotional regulation physi...,2,0.0,0.0,42339210
3,Q_EVAL_004,ambient music sleep quality autonomic nervous ...,2,0.0,0.0,35714120


## 4. Grounded LLM Explanation Evaluation

We evaluate structural JSON validity, citation grounding accuracy, track ID grounding accuracy, and non-clinical safety compliance.

In [5]:
llm_evaluator = LLMExplanationEvaluator()
generator = ExplanationGenerator(mode="DEMO")

# Construct sample request for evaluation
sample_prof = controlled_profiles[0]
sample_recs = catalog_df.head(3).to_dict(orient="records")
sample_chunks = retriever.retrieve("music therapy stress reduction", top_k=2)
from src.rag.evidence import build_evidence_package
sample_evidence = build_evidence_package("music therapy stress reduction", sample_chunks)

req = ExplanationRequest(
    user_profile=sample_prof,
    recommendations=sample_recs,
    acoustic_profiles=sample_prof.get("audio_feature_summary", {}),
    evidence_package=sample_evidence,
    safety_constraints=SafetyConstraints()
)

response = generator.generate(req)
llm_eval_metrics = llm_evaluator.evaluate_explanation(req, response)

print("LLM Explanation Layer Evaluation Results:")
print(json.dumps(llm_eval_metrics, indent=2))

LLM Explanation Layer Evaluation Results:
{
  "json_structural_validity": true,
  "missing_keys": [],
  "citation_grounding_accuracy": 1.0,
  "track_grounding_accuracy": 1.0,
  "safety_compliance_score": 1.0,
  "empty_evidence_compliance_pass": true,
  "malformed_json_recovery_pass": true,
  "overall_grounding_score": 1.0,
  "is_validated": true,
  "validation_warnings_count": 0
}


## 5. End-to-End Pipeline Evaluation

We execute end-to-end benchmark runs across all controlled profile personas, measuring stage-by-stage latency breakdowns and validation pass rates.

In [6]:
e2e_evaluator = EndToEndEvaluator(catalog_df=catalog_df, vector_store=vector_store, embedder=embedder)
e2e_results = e2e_evaluator.evaluate_pipeline(controlled_profiles, top_n_recs=3, top_k_rag=2)

print("End-to-End Pipeline Evaluation Results:")
print(f" - Pipeline Success Rate: {e2e_results['pipeline_success_rate'] * 100:.1f}%")
print(f" - Grounding Validation Pass Rate: {e2e_results['grounding_validation_pass_rate'] * 100:.1f}%")
print(f" - Stage Latency Breakdown (ms):")
for stage, lat in e2e_results["latency_breakdown_ms"].items():
    print(f"    * {stage}: {lat:.2f} ms")

print("\nProfile Execution Breakdown:")
display(pd.DataFrame(e2e_results["profile_execution_details"]))

End-to-End Pipeline Evaluation Results:
 - Pipeline Success Rate: 100.0%
 - Grounding Validation Pass Rate: 100.0%
 - Stage Latency Breakdown (ms):
    * avg_recommendation_ms: 8.58 ms
    * avg_rag_retrieval_ms: 11.53 ms
    * avg_explanation_generation_ms: 0.40 ms
    * avg_total_pipeline_ms: 21.48 ms

Profile Execution Breakdown:


,persona,user_id,status,recommendation_count,retrieved_chunks_count,is_validated,grounding_score,rec_latency_ms,rag_latency_ms,exp_latency_ms,total_latency_ms
0,acoustic_relaxing,USR_EVAL_001,SUCCESS,3,2,True,1.0,10.65,14.11,0.36,26.30
1,upbeat_energetic,USR_EVAL_002,SUCCESS,3,2,True,1.0,7.51,10.11,0.38,18.85
2,ambient_focus,USR_EVAL_003,SUCCESS,3,2,True,1.0,7.57,10.36,0.46,19.29
